# Breast Cancer Detection using GAN and CNN

## Overview
This project uses GAN-generated synthetic images to improve CNN-based breast cancer classification.

# Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     Conv2DTranspose, BatchNormalization, LeakyReLU, Reshape, Input)

# Data Preprocessing

In [ ]:

image_path = "/Users/saad/Desktop/Dissertation/archive (1)/jpeg"
csv_path = "/Users/saad/Desktop/Dissertation/archive (1)/csv/mass_case_description_train_set.csv"

print("Image folder:", image_path)
print("CSV file:", csv_path)

In [ ]:
df = pd.read_csv(csv_path)

df["label"] = df["pathology"].apply(lambda x: 1 if x == "MALIGNANT" else 0)

# Extract UID for mapping
df["uid"] = df["cropped image file path"].apply(lambda x: x.split("/")[-2])

label_map = dict(zip(df["uid"], df["label"]))

print("Total labeled entries:", len(label_map))

In [ ]:
images = []
labels = []
image_size = 224

for folder in os.listdir(image_path):

    folder_path = os.path.join(image_path, folder)

    if not os.path.isdir(folder_path):
        continue

    if folder not in label_map:
        continue

    label = label_map[folder]

    for file in os.listdir(folder_path):
        if file.endswith(".jpg"):

            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                continue

            img = cv2.resize(img, (224, 224))
            img = img / 255.0

            images.append(img)
            labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Total images:", images.shape)
print("Label distribution:", np.bincount(labels))

In [ ]:
images = images.reshape(-1, 224, 224, 1)

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, stratify=labels, random_state=42
)

# Baseline CNN Model

In [ ]:
def build_cnn():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

In [ ]:
cnn_model = build_cnn()

history = cnn_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# Baseline Results and Evaluation

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title("Loss")
plt.legend()

plt.show()

In [ ]:
y_pred_base = (cnn_model.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred_base))

In [ ]:
cm = confusion_matrix(y_test, y_pred_base)

plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Baseline CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Convert images from [0,1] → [-1,1]
gan_images = (X_train * 2) - 1

print("GAN data shape:", gan_images.shape)

# GAN Model

In [ ]:
def build_generator():

    model = Sequential()

    # Input: noise vector (100)
    model.add(Dense(128 * 56 * 56, input_dim=100))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((56, 56, 128)))

    # Upsampling
    model.add(Conv2DTranspose(128, kernel_size=4, strides=2, padding='same'))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2DTranspose(64, kernel_size=4, strides=2, padding='same'))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2))

    # Output (224x224x1)
    model.add(Conv2DTranspose(1, kernel_size=4, padding='same', activation='tanh'))

    return model

generator = build_generator()
generator.summary()

In [ ]:
def build_discriminator():

    model = Sequential()

    model.add(Conv2D(64, kernel_size=3, strides=2, padding='same',
                     input_shape=(224, 224, 1)))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.3))

    model.add(Conv2D(128, kernel_size=3, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.3))

    model.add(Conv2D(256, kernel_size=3, strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Dropout(0.3))

    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))

    return model


discriminator = build_discriminator()

discriminator.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

discriminator.summary()

In [ ]:
# Freeze discriminator
discriminator.trainable = False

noise_dim = 100

gan_input = Input(shape=(noise_dim,))
generated_image = generator(gan_input)
gan_output = discriminator(generated_image)

gan = Model(gan_input, gan_output)

gan.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

gan.summary()

# GAN Training

In [ ]:


epochs = 500
batch_size = 16

real_labels = np.ones((batch_size, 1))
fake_labels = np.zeros((batch_size, 1))

# Store losses
g_losses = []
d_losses = []

for epoch in range(epochs):

    # -----------------------------
    # Train Discriminator
    # -----------------------------
    discriminator.trainable = True

    idx = np.random.randint(0, gan_images.shape[0], batch_size)
    real_images = gan_images[idx]

    noise = np.random.normal(0, 1, (batch_size, noise_dim))
    fake_images = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_images, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

    d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])

    # -----------------------------
    # Train Generator
    # -----------------------------
    discriminator.trainable = False

    noise = np.random.normal(0, 1, (batch_size, noise_dim))
    g_loss = gan.train_on_batch(noise, real_labels)

    # Save losses
    d_losses.append(d_loss)
    g_losses.append(g_loss)

    # -----------------------------
    # Progress
    # -----------------------------
    if epoch % 50 == 0:
        print(f"Epoch {epoch}")
        print(f"D Loss: {d_loss}")
        print(f"G Loss: {g_loss}")
        print("-------------")

In [ ]:

epochs = 5000
batch_size = 32

real_labels = np.ones((batch_size, 1))
fake_labels = np.zeros((batch_size, 1))

for epoch in range(epochs):

    # -----------------------------
    # Train Discriminator
    # -----------------------------
    discriminator.trainable = True

    idx = np.random.randint(0, gan_images.shape[0], batch_size)
    real_images = gan_images[idx]

    noise = np.random.normal(0, 1, (batch_size, noise_dim))
    fake_images = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_images, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

    # -----------------------------
    # Train Generator
    # -----------------------------
    discriminator.trainable = False

    noise = np.random.normal(0, 1, (batch_size, noise_dim))
    g_loss = gan.train_on_batch(noise, real_labels)

    # -----------------------------
    # Progress
    # -----------------------------
    if epoch % 500 == 0:
        print(f"Epoch {epoch}")
        print(f"D Loss: {d_loss_real}")
        print(f"G Loss: {g_loss}")
        print("-------------")

# Synthetic Image Generation

In [ ]:
noise = np.random.normal(0,1,(1000,100))
generated_images = generator.predict(noise)

generated_images = (generated_images + 1) / 2

print("Generated shape:", generated_images.shape)

In [ ]:
plt.figure(figsize=(10,5))

for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(generated_images[i].reshape(224,224), cmap='gray')
    plt.axis('off')

plt.show()

# GAN-Enhanced CNN Model

In [ ]:
y_fake = np.random.randint(0, 2, size=len(generated_images))

# Combine original + synthetic data
X_enhanced = np.concatenate([X_train, generated_images])
y_enhanced = np.concatenate([y_train, y_fake])

print("Enhanced dataset:", X_enhanced.shape)

In [ ]:
cnn_model_enhanced = build_cnn()

cnn_model_enhanced.fit(
    X_enhanced,
    y_enhanced,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# Results and Comparison

In [ ]:
y_pred_enh = (cnn_model_enhanced.predict(X_test) > 0.5).astype(int)

print(classification_report(y_test, y_pred_enh))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "GAN Enhanced"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_base),
        accuracy_score(y_test, y_pred_enh)
    ],
    "Precision": [
        precision_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_enh)
    ],
    "Recall": [
        recall_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_enh)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_enh)
    ]
})

print(comparison)

## Conclusion

The GAN-enhanced model did not improve classification performance. 
Although synthetic images were generated, their quality was low, with many outputs appearing unrealistic or completely black. 

This negatively affected model training, leading to reduced performance compared to the baseline CNN. 

These results demonstrate that GAN-based augmentation is highly dependent on the quality of generated data, and poor-quality synthetic images can degrade model performance rather than improve it.